# Tutorial 11F: Complete Case Study - Brazilian Manufacturing Firms

**Module**: Stochastic Frontier Analysis | **Difficulty**: Advanced | **Duration**: 5-6 hours

**Prerequisites**: All previous tutorials (01-05); Real data analysis experience

## Learning Objectives

1. **Apply** a complete SFA workflow from data exploration to reporting
2. **Integrate** concepts from all previous tutorials
3. **Conduct** rigorous model selection and validation
4. **Analyze** efficiency and productivity patterns
5. **Identify** policy-relevant determinants of inefficiency
6. **Produce** publication-quality outputs (tables, figures, reports)
7. **Interpret** results in economic context
8. **Communicate** findings to stakeholders

## Table of Contents

1. [Setup and Data Loading](#1-setup)
2. [Phase 1: Data Exploration](#2-exploration)
3. [Phase 2: Model Selection](#3-model-selection)
4. [Phase 3: Efficiency Analysis](#4-efficiency)
5. [Phase 4: Determinants of Inefficiency](#5-determinants)
6. [Phase 5: TFP Decomposition](#6-tfp)
7. [Phase 6: Report Generation](#7-reporting)
8. [Exercises](#8-exercises)
9. [Summary and References](#9-summary)

## Research Questions

**Objective**: Analyze efficiency and productivity of Brazilian manufacturing firms (2010-2019)

1. What is the average efficiency level in Brazilian manufacturing?
2. Has efficiency improved or declined over time?
3. What firm characteristics determine inefficiency?
4. Should policies focus on structural reforms or short-term interventions?
5. Are there regional or sectoral differences in efficiency?

## 1. Setup and Data Loading

<a id="1-setup"></a>

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
from pathlib import Path

from panelbox.datasets import load_dataset

np.random.seed(42)
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

BASE_DIR = Path("..")
FIGURES_DIR = BASE_DIR / "outputs" / "figures" / "06_case_study"
TABLES_DIR_LATEX = BASE_DIR / "outputs" / "tables" / "latex"
TABLES_DIR_HTML = BASE_DIR / "outputs" / "tables" / "html"
REPORTS_DIR = BASE_DIR / "outputs" / "reports" / "efficiency_reports"

for d in [FIGURES_DIR, TABLES_DIR_LATEX, TABLES_DIR_HTML, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

In [ ]:
data = load_dataset("brazilian_firms", category="frontier")

print("=" * 80)
print("BRAZILIAN MANUFACTURING FIRMS: EFFICIENCY AND PRODUCTIVITY ANALYSIS")
print("=" * 80)
print(f"\nDataset shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
display(data.head())

## 2. Phase 1: Data Exploration and Preparation

<a id="2-exploration"></a>

### Tasks:
1. Check panel structure (balanced? how many firms/years?)
2. Generate summary statistics for all variables
3. Check for missing values and outliers
4. Create exploratory visualizations

### Key Variables:
- **Output**: `log_output` (log of value added)
- **Inputs**: `log_labor`, `log_capital`, `log_materials`
- **Characteristics**: `firm_age`, `exporter`, `r_and_d`, `foreign_owned`
- **Categorical**: `sector`, `region`, `year`

### 2.1 Panel Structure and Summary Statistics

In [ ]:
# YOUR CODE HERE: Check panel structure
# 1. How many firms? How many years?
# 2. Is the panel balanced?
# 3. What sectors and regions are represented?
#
# Hints:
#   data['firm_id'].nunique()
#   data.groupby('firm_id').size()
#   data['sector'].value_counts()

In [ ]:
# YOUR CODE HERE: Summary statistics
# Descriptive statistics for continuous variables
# Frequency tables for binary variables (exporter, foreign_owned)

### 2.2 Data Quality

In [ ]:
# YOUR CODE HERE: Missing values and outlier detection
# 1. data.isnull().sum()
# 2. For each numeric variable, compute IQR and count outliers:
#    Q1 = data[var].quantile(0.25)
#    Q3 = data[var].quantile(0.75)
#    IQR = Q3 - Q1
#    outliers = ((data[var] < Q1 - 3*IQR) | (data[var] > Q3 + 3*IQR)).sum()

### 2.3 Exploratory Visualizations

In [ ]:
# YOUR CODE HERE: Create a 2x2 figure
# (0,0): Average output over time
# (0,1): Average inputs over time (labor, capital, materials)
# (1,0): Output distribution by sector (boxplot)
# (1,1): Output distribution by region (boxplot)
#
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# ...
# plt.savefig(FIGURES_DIR / 'exploratory_analysis.png', dpi=300, bbox_inches='tight')
# plt.show()

## 3. Phase 2: Model Specification and Selection

<a id="3-model-selection"></a>

Model selection follows a systematic procedure:

1. **Test for inefficiency**: Is there evidence that $\sigma^2_u > 0$?
2. **Select distribution**: Half-normal, exponential, or truncated normal?
3. **Select panel specification**: Pooled, Pitt-Lee, or BC92?
4. **Test returns to scale**: CRS, IRS, or DRS?

### Key API:
```python
model = StochasticFrontier(
    data=data, depvar='log_output',
    exog=['log_labor', 'log_capital', 'log_materials'],
    entity='firm_id', time='year',
    frontier='production', dist='half_normal',
    model_type='pooled'
)
result = model.fit(method='mle', optimizer='L-BFGS-B')
```

### 3.1 Test for Presence of Inefficiency

In [ ]:
# YOUR CODE HERE: Test for inefficiency
# 1. Estimate OLS baseline: sm.OLS(y, sm.add_constant(X)).fit()
# 2. Estimate SFA (pooled, half_normal)
# 3. Run inefficiency_presence_test(
#        loglik_sfa=result.loglik, loglik_ols=ols_result.llf,
#        residuals_ols=ols_result.resid.values,
#        frontier_type='production', distribution='half_normal')
# 4. Interpret: Is sigma_u > 0?

### 3.2 Compare Distributions

In [ ]:
# YOUR CODE HERE: Compare half_normal, exponential, truncated_normal
# Loop over distributions, estimate each, compare AIC/BIC
# Select the best by BIC
#
# distributions = ['half_normal', 'exponential', 'truncated_normal']
# for dist in distributions:
#     model = StochasticFrontier(data=data, ..., dist=dist, model_type='pooled')
#     result = model.fit(method='mle', optimizer='L-BFGS-B')

### 3.3 Select Panel Specification

In [ ]:
# YOUR CODE HERE: Compare Pooled, Pitt-Lee, BC92
# model_type='pooled', 'pitt_lee', 'bc92'
# Compare AIC/BIC, run LR tests between nested models:
#   lr_test(loglik_restricted, loglik_unrestricted, df_diff)

### 3.4 Test Returns to Scale

In [ ]:
# YOUR CODE HERE: Test RTS
# best_model.returns_to_scale_test(input_vars=['log_labor', 'log_capital', 'log_materials'])
# Interpret: CRS (sum=1), IRS (sum>1), DRS (sum<1)

In [ ]:
# YOUR CODE HERE: Print final model selection summary
# Distribution, panel model, RTS, log-likelihood, AIC, BIC
# Print best_model.summary()

## 4. Phase 3: Efficiency Analysis

<a id="4-efficiency"></a>

### Key API:
```python
eff = result.efficiency(estimator='bc', ci_level=0.95, by_period=True)
# Returns DataFrame with 'efficiency' column
```

### 4.1 Efficiency Scores

In [ ]:
# YOUR CODE HERE: Calculate efficiency scores
# 1. eff_df = best_model.efficiency(estimator='bc', ci_level=0.95, by_period=True)
# 2. data_eff = data.copy(); data_eff['te'] = eff_df['efficiency'].values
# 3. Print overall statistics: data_eff['te'].describe()

### 4.2 Temporal, Sectoral, and Regional Patterns

In [ ]:
# YOUR CODE HERE: Analyze patterns
# 1. data_eff.groupby('year')['te'].agg(['mean', 'std'])
# 2. data_eff.groupby('sector')['te'].agg(['mean', 'std', 'count'])
# 3. data_eff.groupby('region')['te'].agg(['mean', 'std', 'count'])
# 4. Top/bottom firms: data_eff.groupby('firm_id')['te'].mean().nlargest(20)

### 4.3 Efficiency Visualizations

In [ ]:
# YOUR CODE HERE: 2x2 visualization
# (0,0): Efficiency histogram with mean line
# (0,1): Efficiency over time with confidence band
# (1,0): Efficiency by sector (horizontal bar)
# (1,1): Efficiency by region (horizontal bar)
#
# plt.savefig(FIGURES_DIR / 'efficiency_analysis.png', dpi=300, bbox_inches='tight')

## 5. Phase 4: Determinants of Inefficiency

<a id="5-determinants"></a>

The BC95 model: $u_{it} \sim N^+(z_{it}'\delta, \sigma^2_u)$

### Key API:
```python
model_bc95 = StochasticFrontier(
    data=data, depvar='log_output',
    exog=['log_labor', 'log_capital', 'log_materials'],
    entity='firm_id', time='year',
    frontier='production', dist='truncated_normal',
    model_type='bc95',
    inefficiency_vars=['firm_age', 'exporter', 'r_and_d', 'foreign_owned']
)
result_bc95 = model_bc95.fit(method='mle', optimizer='L-BFGS-B')
```

### 5.1 BC95 Estimation

In [ ]:
# YOUR CODE HERE: Estimate BC95 with determinants
# 1. Estimate model with inefficiency_vars
# 2. Print summary
# 3. Extract delta parameters (look for 'delta_' in param names)
# 4. Positive delta = increases inefficiency, negative = decreases inefficiency

### 5.2 Marginal Effects and Visualizations

In [ ]:
# YOUR CODE HERE:
# 1. Compute BC95 efficiency
# 2. Create boxplots: exporters vs non-exporters, foreign vs domestic
# 3. Run t-tests for group differences
# 4. Compute correlation between R&D and efficiency
#
# plt.savefig(FIGURES_DIR / 'determinants_analysis.png', dpi=300, bbox_inches='tight')

## 6. Phase 5: Four-Component Model

<a id="6-tfp"></a>

$y_{it} = \alpha + x_{it}'\beta + \mu_i - \eta_i + v_{it} - u_{it}$

### Key API:
```python
model_4c = FourComponentSFA(
    data=data, depvar='log_output',
    exog=['log_labor', 'log_capital', 'log_materials'],
    entity='firm_id', time='year', frontier_type='production'
)
result_4c = model_4c.fit(verbose=True)
te_persistent = result_4c.persistent_efficiency()
te_transient = result_4c.transient_efficiency()
te_overall = result_4c.overall_efficiency()
```

### 6.1 Four-Component Estimation

In [ ]:
# YOUR CODE HERE: Estimate four-component model
# 1. Create and fit FourComponentSFA
# 2. Print variance components: sigma_v, sigma_u, sigma_mu, sigma_eta
# 3. Calculate variance shares
# 4. What % of inefficiency is persistent vs transient?

### 6.2 Persistent vs Transient Efficiency

In [ ]:
# YOUR CODE HERE:
# 1. Get persistent, transient, overall efficiency
# 2. Print summary statistics for each
# 3. Create visualizations:
#    - Variance decomposition pie chart
#    - Persistent vs transient scatter plot
#    - Distribution histograms
#
# plt.savefig(FIGURES_DIR / 'tfp_decomposition.png', dpi=300, bbox_inches='tight')

### 6.3 Policy Implications

In [ ]:
# YOUR CODE HERE: Interpret policy implications
# 1. What share of total inefficiency is persistent vs transient?
# 2. Should policies focus on structural reforms or short-term interventions?

## 7. Phase 6: Report Generation

<a id="7-reporting"></a>

### Key API:
```python
# Export to LaTeX/HTML
latex_str = result.to_latex(caption='Title', label='tab:label')
html_str = result.to_html()
df.to_latex(path, float_format='%.4f')
df.to_html(path, float_format='%.4f')
```

In [ ]:
# YOUR CODE HERE: Export tables
# 1. Export SFA results to LaTeX/HTML
# 2. Export efficiency summary, sectoral, regional tables
# 3. Export model comparison tables

In [ ]:
# YOUR CODE HERE: Generate HTML report
# Create an HTML report summarizing findings
# Save to: REPORTS_DIR / 'brazilian_firms_full_report.html'

In [ ]:
# YOUR CODE HERE: Print summary of all generated outputs
print("=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)

## 8. Exercises

<a id="8-exercises"></a>

### Exercise 1: Robustness Check (Easy)

Re-estimate using a different distribution and compare efficiency rankings.
Use `scipy.stats.spearmanr` for rank correlation. Are rankings robust?

In [ ]:
# Exercise 1: Your code here

### Exercise 2: Sectoral Frontier Analysis (Medium)

Estimate separate frontiers for the two largest sectors.
Compare mean efficiency, variance parameters, and input elasticities.

In [ ]:
# Exercise 2: Your code here

### Exercise 3: BC92 Time-Varying Efficiency (Advanced)

Estimate BC92 model. Extract the eta parameter.
Plot efficiency trajectories for selected firms.

In [ ]:
# Exercise 3: Your code here

## 9. Summary and References

<a id="9-summary"></a>

### Methodology Summary

| Phase | Method | Output |
|-------|--------|--------|
| Data Exploration | EDA | Summary statistics, visualizations |
| Model Selection | LR tests, AIC/BIC | Best specification |
| Efficiency | BC estimator | Firm-level scores |
| Determinants | BC95 model | Policy-relevant factors |
| TFP Decomposition | Four-component | Persistent vs transient |
| Reporting | LaTeX/HTML | Publication-quality outputs |

### References

1. Aigner, D., Lovell, C.A.K., & Schmidt, P. (1977). *JoE*, 6(1), 21-37.
2. Battese, G.E., & Coelli, T.J. (1992). *JPA*, 3, 153-169.
3. Battese, G.E., & Coelli, T.J. (1995). *Empirical Economics*, 20, 325-332.
4. Kumbhakar, S.C., Lien, G., & Hardaker, J.B. (2014). *JPA*, 41(2), 321-337.
5. Kumbhakar, S.C., & Lovell, C.A.K. (2000). *Stochastic Frontier Analysis*. CUP.

---
*PanelBox Stochastic Frontier Analysis module.*